# 09_audit_minutes_pdf_etl — Canonical ETL Freeze (Phase 2)

**목적**: Phase 1 감사(`91_pdf_segment_validation_audit.ipynb`, `gate_decision=READY_FOR_CANONICAL_SEGMENT_EXPORT`)
승인 이후, canonical 산출물을 **이 노트북 하나에서 독립적으로** 생성한다. `91` notebook, 다른 IPYNB, 활성
kernel의 함수·변수·상태는 가져오지 않는다 — 모든 파서·State Machine·검증 로직을 이 노트북 안에서 새로 정의한다.

**이 노트북이 만드는 canonical 산출물** (모두 `data_parse/pdf_crawler/` 밖, 별도 canonical 경로에 저장):
`control_registry.parquet`, `index_map.parquet`, `pages.parquet`, `blocks.parquet`, `speaker_turns.parquet`,
`retrieval_segments.parquet`, `quality_audit.parquet`, `audit_minutes.sqlite`, `pipeline_manifest.json`,
`SSOT_audit_minutes_pdf_etl_v1.0.md`.

TF-IDF, embedding, qrels, PyTorch 학습은 이번 단계에 포함하지 않는다. SVO/Open-IE/skip-gram/NTN/event CNN
관련 설계는 향후 로드맵에서 완전히 제외됐다.


## 00. 실행 계약 · 버전 고정 · 출력 경로

**목적**: `parser_version`/`pipeline_version`을 `1.0.1`로 고정하고, canonical export 활성화, 출력 경로, 환경 정보를 기록한다.
**입력**: 없음.
**출력**: 설정 표, `pipeline_run_id`.
**가능한 실패**: 의존성 미설치(자동 설치 금지).


In [1]:
import sys, os, subprocess, platform, json, re, hashlib, unicodedata, logging, sqlite3
import shutil
from pathlib import Path
from datetime import datetime, timezone, timedelta
from collections import Counter

import pandas as pd
import numpy as np

PROJECT_ROOT = Path("/home/sieg/projects-wsl/SBS_dataScience/DSJA/P3_CULTURE")
NOTEBOOK_PATH = PROJECT_ROOT / "09_audit_minutes_pdf_etl.ipynb"
CANONICAL_ROOT = PROJECT_ROOT / "data_parse" / "audit_minutes_pdf_etl"
LOG_DIR = CANONICAL_ROOT / "logs"

assert PROJECT_ROOT.exists()
CANONICAL_ROOT.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

PARSER_VERSION = "1.0.1"
PIPELINE_VERSION = "1.0.1"
RUN_MODE = "canonical_production"
ALLOW_CANONICAL_EXPORT = True
RUN_OCR = False
RUN_EMBEDDING = False
RUN_LLM = False
RUN_TFIDF = False
RUN_QRELS = False
RUN_PYTORCH = False

RUN_STARTED_UTC = datetime.now(timezone.utc)
RUN_STARTED_KST = RUN_STARTED_UTC.astimezone(timezone(timedelta(hours=9)))
PIPELINE_RUN_ID = f"{PIPELINE_VERSION}_{RUN_STARTED_UTC.strftime('%Y%m%dT%H%M%SZ')}"

logger = logging.getLogger("audit_minutes_pdf_etl")
logger.setLevel(logging.INFO)
logger.handlers.clear()
_fh = logging.FileHandler(LOG_DIR / "audit_minutes_pdf_etl.log", mode="w", encoding="utf-8")
_fh.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
logger.addHandler(_fh)
_sh = logging.StreamHandler()
_sh.setFormatter(logging.Formatter("%(levelname)s %(message)s"))
logger.addHandler(_sh)

def _git(*args):
    try:
        return subprocess.run(["git", *args], cwd=PROJECT_ROOT, capture_output=True,
                               text=True, check=True).stdout.strip()
    except Exception as e:
        return f"UNAVAILABLE({e})"

GIT_BRANCH = _git("branch", "--show-current")
GIT_HEAD = _git("rev-parse", "HEAD")

def _module_version(name):
    try:
        mod = __import__(name)
        return getattr(mod, "__version__", "unknown"), True
    except Exception:
        return None, False

_pymupdf_ver, _pymupdf_ok = _module_version("fitz")
if _pymupdf_ok:
    import fitz
    _pymupdf_ver = getattr(fitz, "pymupdf_version", _pymupdf_ver)
_sklearn_ver, _sklearn_ok = _module_version("sklearn")

SETTINGS_TABLE = {
    "PARSER_VERSION": PARSER_VERSION, "PIPELINE_VERSION": PIPELINE_VERSION, "PIPELINE_RUN_ID": PIPELINE_RUN_ID,
    "RUN_MODE": RUN_MODE, "ALLOW_CANONICAL_EXPORT": ALLOW_CANONICAL_EXPORT,
    "RUN_OCR": RUN_OCR, "RUN_EMBEDDING": RUN_EMBEDDING, "RUN_LLM": RUN_LLM,
    "RUN_TFIDF": RUN_TFIDF, "RUN_QRELS": RUN_QRELS, "RUN_PYTORCH": RUN_PYTORCH,
}
print("=== SETTINGS ===")
for k, v in SETTINGS_TABLE.items():
    print(f"{k}: {v}")

print()
print("=== ENVIRONMENT ===")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"NOTEBOOK_PATH: {NOTEBOOK_PATH}")
print(f"CANONICAL_ROOT: {CANONICAL_ROOT}")
print(f"run_started_utc: {RUN_STARTED_UTC.isoformat()}  run_started_kst: {RUN_STARTED_KST.isoformat()}")
print(f"git_branch: {GIT_BRANCH}  git_head: {GIT_HEAD}")
print(f"python_version: {sys.version.split()[0]}  pandas: {pd.__version__}  numpy: {np.__version__}")
print(f"pymupdf_available: {_pymupdf_ok} version={_pymupdf_ver}")
print(f"sklearn_available: {_sklearn_ok} (INFO only -- not a Phase 2 blocker, TF-IDF not run this phase)")

assert ALLOW_CANONICAL_EXPORT, "this notebook exists to export canonical artifacts"

print()
print("[SECTION 00 RESULT]")
print("status: PASS")
print(f"pipeline_run_id: {PIPELINE_RUN_ID}  parser_version: {PARSER_VERSION}  pipeline_version: {PIPELINE_VERSION}")
print("next: Registry load and integrity re-check")


=== SETTINGS ===
PARSER_VERSION: 1.0.1
PIPELINE_VERSION: 1.0.1
PIPELINE_RUN_ID: 1.0.1_20260721T084913Z
RUN_MODE: canonical_production
ALLOW_CANONICAL_EXPORT: True
RUN_OCR: False
RUN_EMBEDDING: False
RUN_LLM: False
RUN_TFIDF: False
RUN_QRELS: False
RUN_PYTORCH: False

=== ENVIRONMENT ===
PROJECT_ROOT: /home/sieg/projects-wsl/SBS_dataScience/DSJA/P3_CULTURE
NOTEBOOK_PATH: /home/sieg/projects-wsl/SBS_dataScience/DSJA/P3_CULTURE/09_audit_minutes_pdf_etl.ipynb
CANONICAL_ROOT: /home/sieg/projects-wsl/SBS_dataScience/DSJA/P3_CULTURE/data_parse/audit_minutes_pdf_etl
run_started_utc: 2026-07-21T08:49:13.446297+00:00  run_started_kst: 2026-07-21T17:49:13.446297+09:00
git_branch: P3_MARKED_2020_2024  git_head: da09970ba8601f257f34d661fa48afe40573ae1d
python_version: 3.12.3  pandas: 3.0.3  numpy: 2.5.0
pymupdf_available: True version=1.28.0
sklearn_available: False (INFO only -- not a Phase 2 blocker, TF-IDF not run this phase)

[SECTION 00 RESULT]
status: PASS
pipeline_run_id: 1.0.1_20260721T0849

## 01. Registry 로드 및 무결성 재확인

**목적**: `control_registry.parquet`(크롤러 산출물, 읽기 전용)를 로드하고 42행·42 PDF가 여전히 일치하는지 재확인한다.
**입력**: `data_parse/pdf_crawler/control_registry.parquet`, `pdf_raw_data/*.pdf`.
**출력**: registry 통계, PDF integrity 재확인.
**가능한 실패**: 행 수/파일 수 불일치.


In [2]:
REGISTRY_SOURCE_PATH = PROJECT_ROOT / "data_parse" / "pdf_crawler" / "control_registry.parquet"
PDF_DIR = PROJECT_ROOT / "pdf_raw_data"
assert REGISTRY_SOURCE_PATH.exists(), f"not found: {REGISTRY_SOURCE_PATH}"

registry_src = pd.read_parquet(REGISTRY_SOURCE_PATH)
downloaded = registry_src[registry_src["download_status"] == "downloaded"].copy()
actual_pdfs = sorted(PDF_DIR.glob("*.pdf"))

print(f"registry rows: {len(registry_src)}  downloaded rows: {len(downloaded)}  actual pdf files: {len(actual_pdfs)}")
assert len(registry_src) == 42 and len(downloaded) == 42 and len(actual_pdfs) == 42, \
    "registry/pdf count drifted from Phase 1 baseline -- investigate before continuing"

def sha256_of_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

# build canonical control_registry (adds pipeline_run_id / parser_version, parse_status stays untouched
# per the original crawler contract -- this stage does not alter parse_status semantics from Phase 0)
control_registry = registry_src.copy()
control_registry["pipeline_run_id"] = PIPELINE_RUN_ID
control_registry["parser_version"] = PARSER_VERSION

print()
print("[SECTION 01 RESULT]")
print("status: PASS")
print(f"registry_rows: {len(registry_src)}  pdf_files: {len(actual_pdfs)}")
print("next: Block extraction (independent reimplementation)")


registry rows: 42  downloaded rows: 42  actual pdf files: 42

[SECTION 01 RESULT]
status: PASS
registry_rows: 42  pdf_files: 42
next: Block extraction (independent reimplementation)


## 02. Block 추출 (독립 재구현)

**목적**: 90/91 notebook과 동일한 계약(적응형 컬럼 정렬, `PAGE_HEADER`/`TIME_MARKER`/`SPEAKER_HEADER` 정규식,
`block_no` ID 규칙)을 **이 노트북 안에서 새로** 정의한다. Phase 1에서 발견·수정된 `EMPTY`-block 회계
버그는 **처음부터 올바르게** 구현한다(더 이상 "수정"이 아니라 canonical 설계 자체).
**입력**: `pdf_raw_data/*.pdf`.
**출력**: 메모리 상의 `block_df_raw` (canonical `blocks.parquet`의 기반).
**가능한 실패**: 특정 PDF open 실패.


In [3]:
PAGE_HEADER_RE = re.compile(
    r"^(?:\d+\s{0,4})?\d{4}년도국감-문화체육관광\(\d{4}년\d{1,2}월\d{1,2}일\)(?:\s{0,4}\d+)?$"
)
TIME_MARKER_RE = re.compile(
    r"^\(\d{1,2}시\d{1,2}분\s?[^()]{0,20}(개의|개시|속개|중지|계속|종료|산회|정회|폐회)\)$"
)
SPEAKER_HEADER_RE = re.compile(r"^◯")

def ordered_text_blocks(page):
    mid_x = page.rect.width / 2.0
    blocks = [b for b in page.get_text("blocks", sort=True) if b[6] == 0]
    nonempty = [b for b in blocks if b[4].strip()]
    if not nonempty:
        return blocks
    right_ratio = sum(1 for b in nonempty if b[0] > mid_x + 10) / len(nonempty)
    if right_ratio >= 0.15:
        col_of = lambda b: 0 if (b[0] + b[2]) / 2.0 < mid_x else 1
        return sorted(blocks, key=lambda b: (col_of(b), b[1], b[0]))
    return sorted(blocks, key=lambda b: (b[1], b[0]))

def normalized_lines_of_block(raw_text):
    return [ln.strip() for ln in raw_text.split("\n") if ln.strip()]

def classify_block(raw_text):
    lines = normalized_lines_of_block(raw_text)
    if not lines:
        return "EMPTY", "empty_rule"
    first = lines[0]
    if PAGE_HEADER_RE.match(first):
        return "PAGE_HEADER", "PAGE_HEADER_RE"
    if TIME_MARKER_RE.match(first):
        return "TIME_MARKER", "TIME_MARKER_RE"
    if SPEAKER_HEADER_RE.match(first):
        return "SPEAKER_HEADER", "SPEAKER_HEADER_RE"
    return "BODY_TEXT", "fallback_body_text"

def extract_blocks_for_pdf(meeting_id, pdf_path):
    doc = fitz.open(pdf_path)
    rows = []
    page_meta = []
    for pno in range(len(doc)):
        page = doc[pno]
        page_meta.append({"meeting_id": meeting_id, "pdf_page_seq": pno + 1,
                           "page_width": page.rect.width, "page_height": page.rect.height})
        for block_seq, b in enumerate(ordered_text_blocks(page)):
            x0, y0, x1, y1, raw_text, bno, btype = b
            block_no = f"{meeting_id}_{pno+1:04d}_{block_seq:04d}"
            norm_lines = normalized_lines_of_block(raw_text)
            if btype == 1:
                block_type, rule = "IMAGE", "image_block"
            else:
                block_type, rule = classify_block(raw_text)
            rows.append({
                "block_no": block_no, "page_no": f"{meeting_id}_{pno+1:04d}", "meeting_id": meeting_id,
                "index_no": None, "block_seq": block_seq, "x0": x0, "y0": y0, "x1": x1, "y1": y1,
                "source_block_kind": "IMAGE" if btype == 1 else "TEXT",
                "block_text": raw_text, "normalized_text": "".join(norm_lines),
                "block_type": block_type, "classification_rule": rule,
                "classification_confidence": 1.0 if rule != "fallback_body_text" else np.nan,
                "pdf_page_seq": pno + 1,
            })
    doc.close()
    return rows, page_meta

all_block_rows, all_page_meta = [], []
_extract_errors = []
for _, r in downloaded.iterrows():
    mid = r["meeting_id"]
    try:
        rows, page_meta = extract_blocks_for_pdf(mid, PDF_DIR / f"{mid}.pdf")
        all_block_rows.extend(rows)
        all_page_meta.extend(page_meta)
    except Exception as e:
        _extract_errors.append({"meeting_id": mid, "error_type": type(e).__name__, "error_message": str(e)})
        logger.error(f"extraction failed for {mid}: {type(e).__name__}: {e}")

block_df_raw = pd.DataFrame(all_block_rows)
page_meta_df = pd.DataFrame(all_page_meta)
print(f"block_df_raw rows: {len(block_df_raw)}  page_meta rows: {len(page_meta_df)}  extract_errors: {len(_extract_errors)}")

print()
print("[SECTION 02 RESULT]")
print("status:", "PASS" if not _extract_errors else "FAIL")
print(f"block_rows: {len(block_df_raw)}  failed_meetings: {len(_extract_errors)}")
print("next: Page aggregation (pages.parquet)")


block_df_raw rows: 293717  page_meta rows: 4495  extract_errors: 0

[SECTION 02 RESULT]
status: PASS
block_rows: 293717  failed_meetings: 0
next: Page aggregation (pages.parquet)


## 03. Page 집계 (`pages.parquet`)

**목적**: block 단위를 페이지 단위로 집계해 SSOT §6.2 `page_df` 계약을 채운다.
**입력**: `block_df_raw`, `page_meta_df`.
**출력**: `pages_df`.
**가능한 실패**: 좌표 결측으로 image_area_ratio 계산 실패.
**요구사항 확인**: `PAGE_HEADER`는 `normalized_text`(분석용)에서 제외하되 `native_raw_text`(원문 보존)에는 포함한다.
`EMPTY`는 애초에 정규화된 줄이 없어 어느 쪽에도 문자를 보태지 않는다(§11.1 raw 보존 / §11.2 정규화 허용범위).


In [4]:
HANGUL_RE = re.compile(r"[\uAC00-\uD7A3]")
REPLACEMENT_CHAR = "\ufffd"

def page_quality(text_blocks, image_blocks, page_w, page_h):
    native_raw_text = "".join(text_blocks["block_text"])
    normalized_text = "".join(text_blocks.loc[text_blocks["block_type"] != "PAGE_HEADER", "normalized_text"])
    raw_text = native_raw_text  # no OCR this run -> native is the final selected text
    native_char_count = len(native_raw_text.strip())
    selected_char_count = len(raw_text.strip())
    non_ws = [c for c in raw_text if not c.isspace()]
    hangul_ratio = (sum(1 for c in non_ws if HANGUL_RE.match(c)) / len(non_ws)) if non_ws else 0.0
    replacement_ratio = (raw_text.count(REPLACEMENT_CHAR) / len(raw_text)) if raw_text else 0.0
    page_area = max(page_w * page_h, 1.0)
    image_area = sum(max(0.0, (b.x1 - b.x0)) * max(0.0, (b.y1 - b.y0)) for b in image_blocks.itertuples())
    image_area_ratio = min(image_area / page_area, 1.0)
    text_layer_present = native_char_count > 0
    ocr_required = (native_char_count < 30 or not text_layer_present
                    or image_area_ratio >= 0.85 or replacement_ratio >= 0.02)
    quality_score = 0.0 if (ocr_required and not False) else round(1 - replacement_ratio, 4)
    return {
        "native_raw_text": native_raw_text, "ocr_raw_text": None, "raw_text": raw_text,
        "normalized_text": normalized_text, "native_char_count": native_char_count,
        "selected_char_count": selected_char_count, "hangul_ratio": round(hangul_ratio, 4),
        "replacement_char_ratio": round(replacement_ratio, 4), "image_area_ratio": round(image_area_ratio, 4),
        "text_layer_present": text_layer_present, "ocr_required": ocr_required, "ocr_used": False,
        "extraction_method": "native", "quality_score": quality_score, "error_message": None,
    }

page_rows = []
page_meta_by_key = page_meta_df.set_index(["meeting_id", "pdf_page_seq"])
for (meeting_id, pdf_page_seq), grp in block_df_raw.groupby(["meeting_id", "pdf_page_seq"]):
    text_blocks = grp[grp["source_block_kind"] == "TEXT"]
    image_blocks = grp[grp["source_block_kind"] == "IMAGE"]
    meta = page_meta_by_key.loc[(meeting_id, pdf_page_seq)]
    q = page_quality(text_blocks, image_blocks, meta["page_width"], meta["page_height"])
    page_rows.append({
        "page_no": f"{meeting_id}_{pdf_page_seq:04d}", "meeting_id": meeting_id, "pdf_page_seq": int(pdf_page_seq),
        "printed_page_no": None, "index_no": None, **q,
        "pipeline_run_id": PIPELINE_RUN_ID, "parser_version": PARSER_VERSION,
    })

pages_df = pd.DataFrame(page_rows).sort_values(["meeting_id", "pdf_page_seq"]).reset_index(drop=True)
print(f"pages_df rows: {len(pages_df)}")
print(f"ocr_required pages: {int(pages_df['ocr_required'].sum())}  text_layer_present=False pages: "
      f"{int((~pages_df['text_layer_present']).sum())}")

print()
print("[SECTION 03 RESULT]")
print("status: PASS")
print(f"page_rows: {len(pages_df)}")
print("next: Index mapping (index_map.parquet)")


pages_df rows: 4495
ocr_required pages: 0  text_layer_present=False pages: 0

[SECTION 03 RESULT]
status: PASS
page_rows: 4495
next: Index mapping (index_map.parquet)


## 04. 목차 및 Index 매핑 (`index_map.parquet`)

**목적**: SSOT §6.1 `index_df` 계약. 90 notebook 파일럿에서 이미 이 코퍼스에 TOC/목차 패턴
(`INDEX_ENTRY`, dotted leader)이 **실측상 존재하지 않음**을 확인했다(전체 코퍼스 전수 검색 0건).
**입력**: `block_df_raw`.
**출력**: `index_df` — 스키마는 유지하되 0행(§8 "목차가 없으면 index_df는 0행으로 저장하되 스키마는 유지한다").
**가능한 실패**: 목차가 실제로 있는데 놓친 경우 — 이번 실행에서 재검증한다(가정하지 않음).


In [5]:
TOC_DOTTED_RE = re.compile(r"\.{3,}\s*\d+\s*$")
toc_candidates = block_df_raw[block_df_raw["normalized_text"].str.contains(TOC_DOTTED_RE, regex=True, na=False)]
print(f"TOC dotted-leader candidates found in this run: {len(toc_candidates)}")
if len(toc_candidates):
    print(toc_candidates[["meeting_id", "block_no", "normalized_text"]].head(10).to_string(index=False))

INDEX_COLUMNS = ["index_no", "meeting_id", "index_order", "index_label_raw", "index_title_raw",
                  "index_title_normalized", "toc_page_no", "body_page_start_no", "body_page_end_no",
                  "mapping_method", "mapping_confidence", "pipeline_run_id", "parser_version"]
index_df = pd.DataFrame(columns=INDEX_COLUMNS)
print(f"index_df rows: {len(index_df)} (schema preserved, 0 rows -- no TOC pattern found)")

print()
print("[SECTION 04 RESULT]")
print("status: PASS")
print(f"index_rows: {len(index_df)}  toc_candidates_this_run: {len(toc_candidates)}")
print("next: Turn construction (speaker_turns.parquet)")


TOC dotted-leader candidates found in this run: 0
index_df rows: 0 (schema preserved, 0 rows -- no TOC pattern found)

[SECTION 04 RESULT]
status: PASS
index_rows: 0  toc_candidates_this_run: 0
next: Turn construction (speaker_turns.parquet)


## 05. Turn 구축 (`speaker_turns.parquet`)

**목적**: `◯` 화자 헤더 경계로 turn을 구성한다. `EMPTY`는 `BODY_TEXT`/`IMAGE`와 동일한 공통 경로로 처리해서
`n_blocks`/`block_end_no`/`page_end_no` 갱신에서 누락되지 않도록 **처음부터 올바르게** 구현한다
(Phase 1 감사에서 발견된 결함의 canonical 버전은 이 형태로 고정한다).
**입력**: `block_df_raw`.
**출력**: `turn_df_raw`.
**가능한 실패**: 없음(다음 절에서 전수 검증).


In [6]:
def build_turns(block_df):
    turn_rows, turn_seq_by_meeting, current_by_meeting = [], {}, {}
    block_df_sorted = block_df.sort_values(["meeting_id", "pdf_page_seq", "block_seq"]).reset_index(drop=True)
    turn_no_col = [None] * len(block_df_sorted)

    def open_turn(meeting_id, turn_type, speaker_raw, page_no, block_no):
        turn_seq_by_meeting[meeting_id] = turn_seq_by_meeting.get(meeting_id, 0) + 1
        seq = turn_seq_by_meeting[meeting_id]
        t = {
            "turn_no": f"{meeting_id}_T{seq:05d}", "meeting_id": meeting_id, "turn_seq": seq,
            "turn_type": turn_type, "speaker_raw": speaker_raw,
            "page_start_no": page_no, "page_end_no": page_no,
            "block_start_no": block_no, "block_end_no": block_no,
            "block_nos": [], "time_markers": [], "n_speaker_headers": 0,
        }
        turn_rows.append(t)
        current_by_meeting[meeting_id] = t
        return t

    for idx, br in block_df_sorted.iterrows():
        meeting_id = br["meeting_id"]
        bt = br["block_type"]
        current = current_by_meeting.get(meeting_id)
        if bt == "PAGE_HEADER":
            turn_no_col[idx] = None
            continue
        if bt == "TIME_MARKER":
            if current is None:
                current = open_turn(meeting_id, "ORPHAN_FRONT_MATTER", None, br["pdf_page_seq"], br["block_no"])
            current["time_markers"].append(br["normalized_text"])
        elif bt == "SPEAKER_HEADER":
            current = open_turn(meeting_id, "SPEAKER_TURN", br["normalized_text"], br["pdf_page_seq"], br["block_no"])
            current["n_speaker_headers"] += 1
        else:  # BODY_TEXT, EMPTY, IMAGE -- identical bookkeeping path (canonical fix baked in)
            if current is None:
                current = open_turn(meeting_id, "ORPHAN_FRONT_MATTER", None, br["pdf_page_seq"], br["block_no"])
        current["block_nos"].append(br["block_no"])
        current["page_end_no"] = br["pdf_page_seq"]
        current["block_end_no"] = br["block_no"]
        turn_no_col[idx] = current["turn_no"]

    block_df_sorted["turn_no"] = turn_no_col
    norm_by_no = dict(zip(block_df_sorted["block_no"], block_df_sorted["normalized_text"]))
    for t in turn_rows:
        t["raw_text"] = "".join(norm_by_no[b] for b in t["block_nos"])
        t["normalized_text"] = t["raw_text"]
        t["char_count"] = len(t["raw_text"])
        t["n_blocks"] = len(t["block_nos"])
        t["sentence_count"] = len(re.findall(r"다\.|까\?|습니다|입니다|[.!?]", t["raw_text"])) or (1 if t["raw_text"] else 0)
        t["is_orphan"] = t["turn_type"] == "ORPHAN_FRONT_MATTER"
    return block_df_sorted, turn_rows

block_df, _turn_rows = build_turns(block_df_raw)

def _split_speaker(raw):
    if pd.isna(raw) or not isinstance(raw, str) or not raw:
        return None, None, None
    m = re.match(r"^◯(.+?)\s{2,}", raw)
    core = m.group(1) if m else raw[1:]
    role_m = re.search(r"(위원장|위원|장관|차관|처장|청장|국장|과장|실장|대표|증인|참고인|본부장)$", core)
    role = role_m.group(1) if role_m else None
    name = core[: role_m.start()].strip() if role_m else core.strip()
    return core, name, role

for t in _turn_rows:
    core, name, role = _split_speaker(t["speaker_raw"])
    t["speaker_name"] = name
    t["speaker_role"] = role
    t["speaker_org"] = None
    t["speaker_parse_confidence"] = 1.0 if role else (0.5 if core else None)
    t["agenda_text"] = None
    t["time_marker"] = "; ".join(t["time_markers"]) if t["time_markers"] else None
    t["index_no"] = None
    t["parse_confidence"] = 1.0

turn_df_raw = pd.DataFrame([{k: v for k, v in t.items() if k not in ("block_nos", "time_markers", "n_speaker_headers")}
                             for t in _turn_rows])
_turn_block_nos = {t["turn_no"]: t["block_nos"] for t in _turn_rows}
_turn_n_speaker_headers = {t["turn_no"]: t["n_speaker_headers"] for t in _turn_rows}

turn_df_raw["pipeline_run_id"] = PIPELINE_RUN_ID
turn_df_raw["parser_version"] = PARSER_VERSION

print(f"turn_df_raw rows: {len(turn_df_raw)}")
print(turn_df_raw["turn_type"].value_counts().to_dict())

print()
print("[SECTION 05 RESULT]")
print("status: PASS")
print(f"turn_rows: {len(turn_df_raw)}")
print("next: Turn hierarchy exhaustive verification")


turn_df_raw rows: 65590
{'SPEAKER_TURN': 65548, 'ORPHAN_FRONT_MATTER': 42}

[SECTION 05 RESULT]
status: PASS
turn_rows: 65590
next: Turn hierarchy exhaustive verification


## 06. Turn 위계 전수 검증 + EMPTY/PAGE_HEADER/PAGE_FOOTER 비기여 검증

**목적**: Phase 1과 동일한 방법으로 65,590개 turn 전체를 전수 검증하고(0건 기대), 요구사항 #6대로
`EMPTY`/`PAGE_HEADER`/`PAGE_FOOTER`가 turn의 `raw_text`/`normalized_text`/`char_count`에 전혀 기여하지
않았음을 명시적으로 확인한다.
**입력**: `block_df`, `turn_df_raw`.
**출력**: 위계 실패 카운트(0 기대), 비기여 검증 결과.
**가능한 실패**: 하나라도 0이 아니면 canonical export를 진행하지 않는다.


In [7]:
block_sorted = block_df.sort_values(["meeting_id", "pdf_page_seq", "block_seq"]).reset_index(drop=True)
assigned = block_sorted[block_sorted["turn_no"].notna()]
grp = assigned.groupby("turn_no")
actual_n_blocks = grp.size().rename("actual_n_blocks")
actual_first_block = grp["block_no"].first().rename("actual_block_start_no")
actual_last_block = grp["block_no"].last().rename("actual_block_end_no")
actual_page_min = grp["pdf_page_seq"].min().rename("actual_page_start_no")
actual_page_max = grp["pdf_page_seq"].max().rename("actual_page_end_no")
actual_text = grp["normalized_text"].apply(lambda s: "".join(s)).rename("actual_raw_text")
actual_meeting_ids = grp["meeting_id"].apply(lambda s: set(s.unique())).rename("actual_meeting_ids")
actual_first_type = grp["block_type"].first().rename("actual_first_block_type")

joined = turn_df_raw.set_index("turn_no").join(
    [actual_n_blocks, actual_first_block, actual_last_block, actual_page_min, actual_page_max,
     actual_text, actual_meeting_ids, actual_first_type]
)
joined["n_speaker_headers"] = joined.index.map(_turn_n_speaker_headers)

inheritance_fail = int((~joined.apply(lambda r: isinstance(r["actual_meeting_ids"], set)
                                       and r["actual_meeting_ids"] == {r["meeting_id"]}, axis=1)).sum())
n_blocks_mismatch = int((joined["n_blocks"] != joined["actual_n_blocks"]).sum())
char_count_mismatch = int((joined["char_count"] != joined["actual_raw_text"].str.len()).sum())
page_range_mismatch = int(((joined["page_start_no"] != joined["actual_page_start_no"]) |
                            (joined["page_end_no"] != joined["actual_page_end_no"])).sum())
containment_fail = int(((joined["block_start_no"] != joined["actual_block_start_no"]) |
                         (joined["block_end_no"] != joined["actual_block_end_no"])).sum())
multiple_speaker_headers = int(((joined["turn_type"] == "SPEAKER_TURN") & (joined["n_speaker_headers"] > 1)).sum())
speaker_turn_without_header = int(((joined["turn_type"] == "SPEAKER_TURN") &
                                    (joined["actual_first_block_type"] != "SPEAKER_HEADER")).sum())

orphaned_non_header = block_sorted[(block_sorted["turn_no"].isna()) & (block_sorted["block_type"] != "PAGE_HEADER")]
adjacency_fail = len(orphaned_non_header)

def _atomic_check(row):
    parts = row["block_no"].rsplit("_", 2)
    derived_mid = "_".join(parts[:-2])
    try:
        return derived_mid == row["meeting_id"] and int(parts[-2]) == row["pdf_page_seq"] and int(parts[-1]) == row["block_seq"]
    except Exception:
        return False

block_sorted["atomicity_ok"] = block_sorted.apply(_atomic_check, axis=1)
atomicity_fail = int((~block_sorted["atomicity_ok"]).sum())

hierarchy_metrics = {
    "inheritance_fail": inheritance_fail, "containment_fail": containment_fail, "adjacency_fail": adjacency_fail,
    "atomicity_fail": atomicity_fail, "n_blocks_mismatch": n_blocks_mismatch, "char_count_mismatch": char_count_mismatch,
    "page_range_mismatch": page_range_mismatch, "multiple_speaker_headers": multiple_speaker_headers,
    "speaker_turn_without_header": speaker_turn_without_header,
}
print("=== turn hierarchy exhaustive audit ===")
for k, v in hierarchy_metrics.items():
    print(f"{k}: {v}")


=== turn hierarchy exhaustive audit ===
inheritance_fail: 0
containment_fail: 0
adjacency_fail: 0
atomicity_fail: 0
n_blocks_mismatch: 0
char_count_mismatch: 0
page_range_mismatch: 0
multiple_speaker_headers: 0
speaker_turn_without_header: 0


In [8]:
# non-contribution check: for every EMPTY block, normalized_text must be "" (contributes 0 chars);
# for every PAGE_HEADER block, turn_no must be null (never joined into any turn's text at all).
empty_blocks = block_sorted[block_sorted["block_type"] == "EMPTY"]
page_header_blocks = block_sorted[block_sorted["block_type"] == "PAGE_HEADER"]

empty_contributes_chars = int((empty_blocks["normalized_text"].str.len() > 0).sum())
page_header_has_turn = int(page_header_blocks["turn_no"].notna().sum())
page_footer_blocks = block_sorted[block_sorted["block_type"] == "PAGE_FOOTER"] if \
    (block_sorted["block_type"] == "PAGE_FOOTER").any() else block_sorted.iloc[0:0]
page_footer_has_turn = int(page_footer_blocks["turn_no"].notna().sum()) if len(page_footer_blocks) else 0

print(f"EMPTY blocks: {len(empty_blocks)}  contributing >0 chars to normalized_text: {empty_contributes_chars}")
print(f"PAGE_HEADER blocks: {len(page_header_blocks)}  with non-null turn_no (should be 0): {page_header_has_turn}")
print(f"PAGE_FOOTER blocks: {len(page_footer_blocks)}  with non-null turn_no (should be 0): {page_footer_has_turn}")

non_contribution_pass = (empty_contributes_chars == 0 and page_header_has_turn == 0 and page_footer_has_turn == 0)

turn_hierarchy_pass = all(v == 0 for v in hierarchy_metrics.values())

print()
print("[SECTION 06 RESULT]")
print("status:", "PASS" if turn_hierarchy_pass and non_contribution_pass else "FAIL")
for k, v in hierarchy_metrics.items():
    print(f"{k}: {v}")
print(f"non_contribution_pass: {non_contribution_pass}")
print("next: Retrieval Segment construction")


EMPTY blocks: 2  contributing >0 chars to normalized_text: 0
PAGE_HEADER blocks: 4495  with non-null turn_no (should be 0): 0
PAGE_FOOTER blocks: 0  with non-null turn_no (should be 0): 0

[SECTION 06 RESULT]
status: PASS
inheritance_fail: 0
containment_fail: 0
adjacency_fail: 0
atomicity_fail: 0
n_blocks_mismatch: 0
char_count_mismatch: 0
page_range_mismatch: 0
multiple_speaker_headers: 0
speaker_turn_without_header: 0
non_contribution_pass: True
next: Retrieval Segment construction


## 07. Retrieval Segment 구축 (`retrieval_segments.parquet`)

**목적**: `turn_df_raw`를 `meeting_id`·`turn_seq` 순으로 정렬해 `SEG_TURN`/`SEG_PREV_CURR`/`SEG_CURR_NEXT`
3종을 생성한다. 경계 정책은 Phase 1과 동일한 `omit_missing_neighbor`.
**입력**: `turn_df_raw`.
**출력**: `segment_df_raw`.
**가능한 실패**: 없음(다음 절에서 검증).


In [9]:
turn_sorted2 = turn_df_raw.sort_values(["meeting_id", "turn_seq"]).reset_index(drop=True)

def _seg_hash(meeting_id, segment_type, turn_nos, normalized_text):
    payload = f"{meeting_id}|{segment_type}|{'|'.join(turn_nos)}|{normalized_text}"
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

segment_rows = []
for meeting_id, grp in turn_sorted2.groupby("meeting_id"):
    grp = grp.reset_index(drop=True)
    n = len(grp)
    for i in range(n):
        row = grp.iloc[i]
        anchor_no = row["turn_no"]
        anchor_seq = int(row["turn_seq"])
        prev_row = grp.iloc[i - 1] if i > 0 else None
        next_row = grp.iloc[i + 1] if i < n - 1 else None

        seg_text = row["normalized_text"]
        segment_rows.append({
            "segment_no": f"{meeting_id}_SEG_TURN_{anchor_seq:05d}", "meeting_id": meeting_id, "index_no": None,
            "segment_type": "SEG_TURN", "anchor_turn_no": anchor_no,
            "turn_start_no": anchor_no, "turn_end_no": anchor_no, "prev_turn_no": None, "next_turn_no": None,
            "page_start_no": int(row["page_start_no"]), "page_end_no": int(row["page_end_no"]),
            "speaker_names": row["speaker_raw"], "speaker_roles": row["speaker_role"],
            "raw_text": seg_text, "normalized_text": seg_text, "char_count": len(seg_text),
            "sentence_count": int(row["sentence_count"]),
        })
        if prev_row is not None:
            text = prev_row["normalized_text"] + row["normalized_text"]
            segment_rows.append({
                "segment_no": f"{meeting_id}_SEG_PREV_CURR_{anchor_seq:05d}", "meeting_id": meeting_id, "index_no": None,
                "segment_type": "SEG_PREV_CURR", "anchor_turn_no": anchor_no,
                "turn_start_no": prev_row["turn_no"], "turn_end_no": anchor_no,
                "prev_turn_no": prev_row["turn_no"], "next_turn_no": None,
                "page_start_no": int(prev_row["page_start_no"]), "page_end_no": int(row["page_end_no"]),
                "speaker_names": f"{prev_row['speaker_raw']} | {row['speaker_raw']}",
                "speaker_roles": f"{prev_row['speaker_role']} | {row['speaker_role']}",
                "raw_text": text, "normalized_text": text, "char_count": len(text),
                "sentence_count": int(prev_row["sentence_count"]) + int(row["sentence_count"]),
            })
        if next_row is not None:
            text = row["normalized_text"] + next_row["normalized_text"]
            segment_rows.append({
                "segment_no": f"{meeting_id}_SEG_CURR_NEXT_{anchor_seq:05d}", "meeting_id": meeting_id, "index_no": None,
                "segment_type": "SEG_CURR_NEXT", "anchor_turn_no": anchor_no,
                "turn_start_no": anchor_no, "turn_end_no": next_row["turn_no"],
                "prev_turn_no": None, "next_turn_no": next_row["turn_no"],
                "page_start_no": int(row["page_start_no"]), "page_end_no": int(next_row["page_end_no"]),
                "speaker_names": f"{row['speaker_raw']} | {next_row['speaker_raw']}",
                "speaker_roles": f"{row['speaker_role']} | {next_row['speaker_role']}",
                "raw_text": text, "normalized_text": text, "char_count": len(text),
                "sentence_count": int(row["sentence_count"]) + int(next_row["sentence_count"]),
            })

segment_df_raw = pd.DataFrame(segment_rows)
segment_df_raw["segment_hash"] = segment_df_raw.apply(
    lambda r: _seg_hash(r["meeting_id"], r["segment_type"], [r["turn_start_no"], r["turn_end_no"]], r["normalized_text"]),
    axis=1,
)
segment_df_raw["tfidf_ready"] = segment_df_raw["char_count"] > 0
segment_df_raw["pipeline_run_id"] = PIPELINE_RUN_ID
segment_df_raw["parser_version"] = PARSER_VERSION

print(f"segment_df_raw rows: {len(segment_df_raw)}")
print(segment_df_raw["segment_type"].value_counts().to_dict())

print()
print("[SECTION 07 RESULT]")
print("status: PASS")
print(f"segment_rows: {len(segment_df_raw)}")
print("next: Segment integrity verification")


segment_df_raw rows: 196686
{'SEG_TURN': 65590, 'SEG_CURR_NEXT': 65548, 'SEG_PREV_CURR': 65548}

[SECTION 07 RESULT]
status: PASS
segment_rows: 196686
next: Segment integrity verification


## 08. Segment 무결성 검증

**목적**: segment PK 유일성, turn FK 무결성, 기대 수(`3n-2`) 대비 실제 수, 원문 역추적률(목표 1.00)을 검증한다.
**입력**: `segment_df_raw`, `turn_df_raw`.
**출력**: `segment_fk_failures`, `traceability_rate`, expected vs observed.
**가능한 실패**: 하나라도 기준 미달이면 canonical export를 진행하지 않는다.


In [10]:
segment_no_dupes = int(segment_df_raw["segment_no"].duplicated().sum())
valid_turn_set = set(turn_df_raw["turn_no"])

def _fk_orphan_count(col):
    vals = segment_df_raw[col].dropna()
    return int((~vals.isin(valid_turn_set)).sum())

fk_orphans = {c: _fk_orphan_count(c) for c in ["anchor_turn_no", "turn_start_no", "turn_end_no", "prev_turn_no", "next_turn_no"]}
segment_fk_failures = sum(fk_orphans.values())

n_per_meeting = turn_df_raw.groupby("meeting_id").size()
expected_per_meeting = n_per_meeting.map(lambda n: 1 if n == 1 else 3 * n - 2)
observed_per_meeting = segment_df_raw.groupby("meeting_id").size()
total_expected = int(expected_per_meeting.sum())
total_observed = int(observed_per_meeting.sum())

turn_text_by_no = turn_df_raw.set_index("turn_no")["normalized_text"]
def _assembly_ok(row):
    try:
        if row["segment_type"] == "SEG_TURN":
            expected = turn_text_by_no[row["anchor_turn_no"]]
        elif row["segment_type"] == "SEG_PREV_CURR":
            expected = turn_text_by_no[row["prev_turn_no"]] + turn_text_by_no[row["anchor_turn_no"]]
        else:
            expected = turn_text_by_no[row["anchor_turn_no"]] + turn_text_by_no[row["next_turn_no"]]
        return expected == row["normalized_text"]
    except KeyError:
        return False

segment_df_raw["assembly_ok"] = segment_df_raw.apply(_assembly_ok, axis=1)
assembly_fail_count = int((~segment_df_raw["assembly_ok"]).sum())

def _traceable(row):
    refs = [row["anchor_turn_no"], row["turn_start_no"], row["turn_end_no"]]
    if pd.notna(row["prev_turn_no"]): refs.append(row["prev_turn_no"])
    if pd.notna(row["next_turn_no"]): refs.append(row["next_turn_no"])
    return all(r in valid_turn_set for r in refs)

segment_df_raw["traceable"] = segment_df_raw.apply(_traceable, axis=1)
traceability_rate = float(segment_df_raw["traceable"].mean())

print(f"segment_no duplicates: {segment_no_dupes}")
print(f"FK orphans: {fk_orphans}  total: {segment_fk_failures}")
print(f"expected_segment_count: {total_expected}  observed_segment_count: {total_observed}")
print(f"assembly_fail_count: {assembly_fail_count}")
print(f"traceability_rate: {traceability_rate}")

segment_integrity_pass = (segment_no_dupes == 0 and segment_fk_failures == 0 and total_expected == total_observed
                           and assembly_fail_count == 0 and traceability_rate == 1.0)

print()
print("[SECTION 08 RESULT]")
print("status:", "PASS" if segment_integrity_pass else "FAIL")
print(f"segment_fk_failures: {segment_fk_failures}  traceability_rate: {traceability_rate}  "
      f"expected==observed: {total_expected == total_observed}")
print("next: Quality audit aggregation")


segment_no duplicates: 0
FK orphans: {'anchor_turn_no': 0, 'turn_start_no': 0, 'turn_end_no': 0, 'prev_turn_no': 0, 'next_turn_no': 0}  total: 0
expected_segment_count: 196686  observed_segment_count: 196686
assembly_fail_count: 0
traceability_rate: 1.0

[SECTION 08 RESULT]
status: PASS
segment_fk_failures: 0  traceability_rate: 1.0  expected==observed: True
next: Quality audit aggregation


## 09. Quality Audit 집계 (`quality_audit.parquet`)

**목적**: SSOT §6.6 `quality_df` 계약(`pipeline_run_id`, `meeting_id`, `metric_name`, `metric_value`,
`threshold`, `status`, `detail`)에 맞춰 전역 지표 + meeting별 page coverage 지표를 하나의 표로 만든다.
**입력**: 앞선 모든 절의 계산 결과.
**출력**: `quality_audit_df`.
**가능한 실패**: 없음(집계 전용).


In [11]:
def qrow(meeting_id, metric_name, metric_value, threshold, status, detail):
    return {"pipeline_run_id": PIPELINE_RUN_ID, "meeting_id": meeting_id, "metric_name": metric_name,
            "metric_value": metric_value, "threshold": threshold, "status": status, "detail": detail}

quality_rows = []
quality_rows.append(qrow("ALL", "registry_rows", len(registry_src), "==42", "PASS" if len(registry_src)==42 else "FAIL", ""))
quality_rows.append(qrow("ALL", "pdf_files", len(actual_pdfs), "==42", "PASS" if len(actual_pdfs)==42 else "FAIL", ""))
quality_rows.append(qrow("ALL", "page_rows", len(pages_df), "==4495", "PASS" if len(pages_df)==4495 else "FAIL", ""))
quality_rows.append(qrow("ALL", "block_rows", len(block_df), "==293717", "PASS" if len(block_df)==293717 else "FAIL", ""))
quality_rows.append(qrow("ALL", "turn_rows", len(turn_df_raw), "==65590", "PASS" if len(turn_df_raw)==65590 else "FAIL", ""))
quality_rows.append(qrow("ALL", "segment_rows", len(segment_df_raw), "==196686", "PASS" if len(segment_df_raw)==196686 else "FAIL", ""))
for k, v in hierarchy_metrics.items():
    quality_rows.append(qrow("ALL", f"turn_{k}", v, "==0", "PASS" if v == 0 else "FAIL", ""))
quality_rows.append(qrow("ALL", "segment_fk_failures", segment_fk_failures, "==0", "PASS" if segment_fk_failures==0 else "FAIL", ""))
quality_rows.append(qrow("ALL", "segment_traceability_rate", traceability_rate, "==1.0", "PASS" if traceability_rate==1.0 else "FAIL", ""))
quality_rows.append(qrow("ALL", "expected_vs_observed_segments", total_observed - total_expected, "==0",
                          "PASS" if total_expected==total_observed else "FAIL", ""))
quality_rows.append(qrow("ALL", "empty_block_noncontribution", empty_contributes_chars, "==0",
                          "PASS" if empty_contributes_chars==0 else "FAIL", ""))
quality_rows.append(qrow("ALL", "page_header_noncontribution", page_header_has_turn, "==0",
                          "PASS" if page_header_has_turn==0 else "FAIL", ""))
quality_rows.append(qrow("ALL", "sklearn_available", int(_sklearn_ok), "n/a", "INFO",
                          "not a Phase 2 blocker -- TF-IDF/embedding not run this phase"))

# per-meeting page coverage
expected_pages_per_meeting = registry_src.set_index("meeting_id")["page_count"]
actual_pages_per_meeting = pages_df.groupby("meeting_id").size()
for mid in registry_src["meeting_id"]:
    exp = int(expected_pages_per_meeting.get(mid, 0))
    act = int(actual_pages_per_meeting.get(mid, 0))
    quality_rows.append(qrow(mid, "page_coverage", act, f"=={exp}", "PASS" if act == exp else "FAIL", ""))

quality_audit_df = pd.DataFrame(quality_rows)
quality_audit_df["parser_version"] = PARSER_VERSION
print(f"quality_audit_df rows: {len(quality_audit_df)}")
print(quality_audit_df["status"].value_counts().to_dict())
core_fail_count = int((quality_audit_df["status"] == "FAIL").sum())
print(f"core_fail_count: {core_fail_count}")

print()
print("[SECTION 09 RESULT]")
print("status:", "PASS" if core_fail_count == 0 else "FAIL")
print(f"quality_rows: {len(quality_audit_df)}  core_fail_count: {core_fail_count}")
print("next: Canonical export (parquet/sqlite/manifest/SSOT)")


quality_audit_df rows: 63
{'PASS': 62, 'INFO': 1}
core_fail_count: 0

[SECTION 09 RESULT]
status: PASS
quality_rows: 63  core_fail_count: 0
next: Canonical export (parquet/sqlite/manifest/SSOT)


## 10. Canonical Export

**목적**: `core_fail_count == 0`일 때만 canonical 산출물을 실제로 기록한다. 모든 parquet/sqlite에
`pipeline_run_id`/`parser_version` 컬럼을 공유해서 넣는다.
**입력**: 위에서 만든 모든 canonical DataFrame.
**출력**: 7개 parquet, 1개 sqlite, `pipeline_manifest.json`, `SSOT_audit_minutes_pdf_etl_v1.0.md`.
**가능한 실패**: `core_fail_count > 0`이면 export를 중단하고 FAIL로 기록한다(품질 실패 행을 숨기거나 삭제하지 않는다).


In [12]:
CANONICAL_PATHS = {
    "control_registry": CANONICAL_ROOT / "control_registry.parquet",
    "index_map": CANONICAL_ROOT / "index_map.parquet",
    "pages": CANONICAL_ROOT / "pages.parquet",
    "blocks": CANONICAL_ROOT / "blocks.parquet",
    "speaker_turns": CANONICAL_ROOT / "speaker_turns.parquet",
    "retrieval_segments": CANONICAL_ROOT / "retrieval_segments.parquet",
    "quality_audit": CANONICAL_ROOT / "quality_audit.parquet",
}
SQLITE_PATH = CANONICAL_ROOT / "audit_minutes.sqlite"
MANIFEST_PATH = CANONICAL_ROOT / "pipeline_manifest.json"
SSOT_PATH = PROJECT_ROOT / "SSOT_audit_minutes_pdf_etl_v1.0.md"

if core_fail_count > 0:
    print("core_fail_count > 0 -- CANONICAL EXPORT ABORTED. Quality failures are NOT deleted or hidden;")
    print("see quality_audit_df for the FAIL rows.")
    print(quality_audit_df[quality_audit_df["status"] == "FAIL"].to_string(index=False))
else:
    blocks_out = block_df.drop(columns=["atomicity_ok"], errors="ignore").copy()
    blocks_out["pipeline_run_id"] = PIPELINE_RUN_ID
    blocks_out["parser_version"] = PARSER_VERSION
    blocks_out = blocks_out[["block_no", "page_no", "meeting_id", "index_no", "block_seq", "x0", "y0", "x1", "y1",
                              "source_block_kind", "block_text", "normalized_text", "block_type",
                              "classification_rule", "classification_confidence", "turn_no",
                              "pipeline_run_id", "parser_version"]].copy()
    blocks_out["is_orphan"] = blocks_out["turn_no"].isna() & (blocks_out["block_type"] != "PAGE_HEADER")

    turns_out = turn_df_raw[["turn_no", "meeting_id", "index_no", "turn_seq", "turn_type", "speaker_raw",
                              "speaker_name", "speaker_role", "speaker_org", "speaker_parse_confidence",
                              "agenda_text", "time_marker", "page_start_no", "page_end_no", "block_start_no",
                              "block_end_no", "raw_text", "normalized_text", "char_count", "sentence_count",
                              "is_orphan", "parse_confidence", "pipeline_run_id", "parser_version"]].copy()

    segments_out = segment_df_raw.drop(columns=["assembly_ok", "traceable"], errors="ignore")[
        ["segment_no", "meeting_id", "index_no", "segment_type", "anchor_turn_no", "turn_start_no", "turn_end_no",
         "prev_turn_no", "next_turn_no", "page_start_no", "page_end_no", "speaker_names", "speaker_roles",
         "raw_text", "normalized_text", "char_count", "sentence_count", "segment_hash", "tfidf_ready",
         "pipeline_run_id", "parser_version"]].copy()

    control_registry.to_parquet(CANONICAL_PATHS["control_registry"], index=False)
    index_df.to_parquet(CANONICAL_PATHS["index_map"], index=False)
    pages_df.to_parquet(CANONICAL_PATHS["pages"], index=False)
    blocks_out.to_parquet(CANONICAL_PATHS["blocks"], index=False)
    turns_out.to_parquet(CANONICAL_PATHS["speaker_turns"], index=False)
    segments_out.to_parquet(CANONICAL_PATHS["retrieval_segments"], index=False)
    quality_audit_df.to_parquet(CANONICAL_PATHS["quality_audit"], index=False)
    for name, p in CANONICAL_PATHS.items():
        print(f"saved: {p} ({p.stat().st_size} bytes)")

    if SQLITE_PATH.exists():
        SQLITE_PATH.unlink()
    conn = sqlite3.connect(SQLITE_PATH)
    control_registry.to_sql("control_registry", conn, index=False)
    index_df.to_sql("index_map", conn, index=False)
    pages_df.to_sql("pages", conn, index=False)
    blocks_out.to_sql("blocks", conn, index=False)
    turns_out.to_sql("speaker_turns", conn, index=False)
    segments_out.to_sql("retrieval_segments", conn, index=False)
    quality_audit_df.to_sql("quality_audit", conn, index=False)
    conn.commit()
    conn.close()
    print(f"saved: {SQLITE_PATH} ({SQLITE_PATH.stat().st_size} bytes)")


saved: /home/sieg/projects-wsl/SBS_dataScience/DSJA/P3_CULTURE/data_parse/audit_minutes_pdf_etl/control_registry.parquet (21898 bytes)
saved: /home/sieg/projects-wsl/SBS_dataScience/DSJA/P3_CULTURE/data_parse/audit_minutes_pdf_etl/index_map.parquet (6623 bytes)
saved: /home/sieg/projects-wsl/SBS_dataScience/DSJA/P3_CULTURE/data_parse/audit_minutes_pdf_etl/pages.parquet (27455903 bytes)
saved: /home/sieg/projects-wsl/SBS_dataScience/DSJA/P3_CULTURE/data_parse/audit_minutes_pdf_etl/blocks.parquet (23294000 bytes)
saved: /home/sieg/projects-wsl/SBS_dataScience/DSJA/P3_CULTURE/data_parse/audit_minutes_pdf_etl/speaker_turns.parquet (22374049 bytes)
saved: /home/sieg/projects-wsl/SBS_dataScience/DSJA/P3_CULTURE/data_parse/audit_minutes_pdf_etl/retrieval_segments.parquet (50828993 bytes)
saved: /home/sieg/projects-wsl/SBS_dataScience/DSJA/P3_CULTURE/data_parse/audit_minutes_pdf_etl/quality_audit.parquet (6368 bytes)


saved: /home/sieg/projects-wsl/SBS_dataScience/DSJA/P3_CULTURE/data_parse/audit_minutes_pdf_etl/audit_minutes.sqlite (497930240 bytes)


In [13]:
if core_fail_count == 0:
    RUN_FINISHED_UTC = datetime.now(timezone.utc)
    pipeline_manifest = {
        "project": "P3_CULTURE", "stage": "CANONICAL_ETL_FREEZE", "notebook": str(NOTEBOOK_PATH.relative_to(PROJECT_ROOT)),
        "pipeline_run_id": PIPELINE_RUN_ID, "parser_version": PARSER_VERSION, "pipeline_version": PIPELINE_VERSION,
        "run_started_utc": RUN_STARTED_UTC.isoformat(), "run_finished_utc": RUN_FINISHED_UTC.isoformat(),
        "git_branch": GIT_BRANCH, "git_head": GIT_HEAD,
        "row_counts": {
            "control_registry": len(control_registry), "index_map": len(index_df), "pages": len(pages_df),
            "blocks": len(blocks_out), "speaker_turns": len(turns_out), "retrieval_segments": len(segments_out),
            "quality_audit": len(quality_audit_df),
        },
        "hierarchy_metrics": hierarchy_metrics,
        "segment_metrics": {
            "expected_segment_count": total_expected, "observed_segment_count": total_observed,
            "segment_fk_failures": segment_fk_failures, "traceability_rate": traceability_rate,
            "assembly_fail_count": assembly_fail_count,
        },
        "non_contribution_check": {
            "empty_block_chars_contributed": empty_contributes_chars,
            "page_header_with_turn_no": page_header_has_turn, "page_footer_with_turn_no": page_footer_has_turn,
        },
        "source_pdf_hashes": dict(zip(registry_src["meeting_id"], registry_src["sha256"])),
        "core_fail_count": core_fail_count,
        "sklearn_available": _sklearn_ok,
        "outputs": {k: str(p.relative_to(PROJECT_ROOT)) for k, p in CANONICAL_PATHS.items()},
        "sqlite": str(SQLITE_PATH.relative_to(PROJECT_ROOT)),
        "ssot_doc": str(SSOT_PATH.relative_to(PROJECT_ROOT)),
    }
    with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
        json.dump(pipeline_manifest, f, ensure_ascii=False, indent=2, default=str)
    print(f"saved: {MANIFEST_PATH}")

    ssot_lines = [
        "# 국정감사 문화체육관광위원회 회의록 PDF ETL SSOT",
        "",
        f"- SSOT 버전: 1.0 (parser_version/pipeline_version={PARSER_VERSION})",
        f"- 실행 상태: EXECUTED (pipeline_run_id={PIPELINE_RUN_ID})",
        f"- 기준 노트북: `09_audit_minutes_pdf_etl.ipynb`",
        f"- 기준 경로: `{PROJECT_ROOT}`",
        "",
        "## 1. 확정 규모",
        f"- registry={len(control_registry)}, pages={len(pages_df)}, blocks={len(blocks_out)}, "
        f"turns={len(turns_out)}, segments={len(segments_out)}",
        "",
        "## 2. 확정 계약",
        "- block_no PK: `{meeting_id}_{page_no:04d}_{block_seq:04d}`",
        "- turn_no PK: `{meeting_id}_T{turn_seq:05d}`",
        "- segment_no PK: `{meeting_id}_{SEG_TYPE}_{anchor_turn_seq:05d}`",
        "- EMPTY/PAGE_HEADER/PAGE_FOOTER는 turn raw_text/normalized_text/char_count에 기여하지 않는다(§06 검증, 0건).",
        "- 이 코퍼스는 TOC/index 패턴이 없어 `index_map.parquet`은 스키마만 유지된 0행이다.",
        "",
        "## 3. Quality Gate",
        f"- core_fail_count: {core_fail_count}",
        f"- hierarchy_metrics: {hierarchy_metrics}",
        f"- segment traceability_rate: {traceability_rate}",
        "",
        "## 4. 로드맵에서 제외된 항목",
        "- SVO, Open-IE, skip-gram, NTN, event CNN 관련 설계·구현은 향후 production/실험 로드맵에서 완전히 제외됐다.",
        "",
        "## 5. 다음 단계",
        "- Phase 3: 긴 turn/공백 소실 대응은 canonical raw/normalized_text를 건드리지 않고 별도 "
        "`retrieval_search_chunks.parquet`(검색 전용)로 분리한다.",
        "- Phase 4: `20_target_segment_sparse_retrieval.ipynb`에서 sparse retrieval production을 시작한다.",
    ]
    SSOT_PATH.write_text("\n".join(ssot_lines), encoding="utf-8")
    print(f"saved: {SSOT_PATH}")

print()
print("[SECTION 10 RESULT]")
print("status:", "PASS" if core_fail_count == 0 else "FAIL")
print(f"canonical_exported: {core_fail_count == 0}")
print("next: Reload verification")


saved: /home/sieg/projects-wsl/SBS_dataScience/DSJA/P3_CULTURE/data_parse/audit_minutes_pdf_etl/pipeline_manifest.json
saved: /home/sieg/projects-wsl/SBS_dataScience/DSJA/P3_CULTURE/SSOT_audit_minutes_pdf_etl_v1.0.md

[SECTION 10 RESULT]
status: PASS
canonical_exported: True
next: Reload verification


## 11. 재로딩 검증

**목적**: 방금 저장한 parquet·sqlite를 다시 읽어서 row count, dtype, PK/FK, `pipeline_run_id`,
`parser_version`이 메모리 상 값과 일치하는지 검산한다.
**입력**: `data_parse/audit_minutes_pdf_etl/*.parquet`, `audit_minutes.sqlite`.
**출력**: 재로딩 검증 결과.
**가능한 실패**: row count/dtype/PK/FK 불일치, run_id/parser_version 불일치.


In [14]:
reload_checks = {}

_reloaded = {name: pd.read_parquet(p) for name, p in CANONICAL_PATHS.items()}
_expected_len = {"control_registry": len(control_registry), "index_map": len(index_df), "pages": len(pages_df),
                 "blocks": len(blocks_out), "speaker_turns": len(turns_out),
                 "retrieval_segments": len(segments_out), "quality_audit": len(quality_audit_df)}
for name, df in _reloaded.items():
    reload_checks[f"{name}_row_count"] = (len(df) == _expected_len[name])

# dtype spot-check (object/string/category/int/float categories, not exact pandas dtype string match)
reload_checks["blocks_dtype_x0_numeric"] = pd.api.types.is_numeric_dtype(_reloaded["blocks"]["x0"])
reload_checks["turns_dtype_char_count_numeric"] = pd.api.types.is_numeric_dtype(_reloaded["speaker_turns"]["char_count"])
reload_checks["segments_dtype_char_count_numeric"] = pd.api.types.is_numeric_dtype(_reloaded["retrieval_segments"]["char_count"])

# PK dup check on reloaded data
reload_checks["blocks_pk_unique"] = _reloaded["blocks"]["block_no"].is_unique
reload_checks["turns_pk_unique"] = _reloaded["speaker_turns"]["turn_no"].is_unique
reload_checks["segments_pk_unique"] = _reloaded["retrieval_segments"]["segment_no"].is_unique

# FK check on reloaded data
_valid_turns_reload = set(_reloaded["speaker_turns"]["turn_no"])
reload_checks["segments_fk_anchor_valid"] = _reloaded["retrieval_segments"]["anchor_turn_no"].isin(_valid_turns_reload).all()
_valid_blocks_reload = set(_reloaded["blocks"]["block_no"])
_non_null_turn_fk = _reloaded["blocks"]["turn_no"].dropna()
reload_checks["blocks_turn_fk_valid"] = _non_null_turn_fk.isin(_valid_turns_reload).all()

# run_id / parser_version consistency across ALL reloaded tables
for name, df in _reloaded.items():
    if "pipeline_run_id" in df.columns:
        reload_checks[f"{name}_run_id_consistent"] = (df["pipeline_run_id"] == PIPELINE_RUN_ID).all()
    if "parser_version" in df.columns:
        reload_checks[f"{name}_parser_version_consistent"] = (df["parser_version"] == PARSER_VERSION).all()

# sqlite reload check
conn = sqlite3.connect(SQLITE_PATH)
for tbl, expected_len in _expected_len.items():
    n = pd.read_sql(f"SELECT COUNT(*) as n FROM {tbl}", conn)["n"].iloc[0]
    reload_checks[f"sqlite_{tbl}_row_count"] = (n == expected_len)
conn.close()

# manifest reload check
with open(MANIFEST_PATH, encoding="utf-8") as f:
    _mf = json.load(f)
reload_checks["manifest_pipeline_run_id_match"] = (_mf["pipeline_run_id"] == PIPELINE_RUN_ID)
reload_checks["manifest_parser_version_match"] = (_mf["parser_version"] == PARSER_VERSION)
reload_checks["manifest_row_counts_match"] = (_mf["row_counts"] == _expected_len)

n_fail = sum(1 for v in reload_checks.values() if not v)
print(f"reload checks: {len(reload_checks)}  failed: {n_fail}")
for k, v in reload_checks.items():
    if not v:
        print(f"  FAIL: {k}")
if n_fail == 0:
    print("all reload checks passed")

print()
print("[SECTION 11 RESULT]")
print("status:", "PASS" if n_fail == 0 else "FAIL")
print(f"reload_checks_total: {len(reload_checks)}  reload_checks_failed: {n_fail}")
print("next: Final canonical ETL gate")


reload checks: 39  failed: 0
all reload checks passed

[SECTION 11 RESULT]
status: PASS
reload_checks_total: 39  reload_checks_failed: 0
next: Final canonical ETL gate


## 12. 최종 판정

**목적**: `CANONICAL_ETL_PASS` / `CONDITIONAL_PASS` / `FAIL` 중 하나로 최종 판정한다.
**입력**: 앞선 모든 절의 결과.
**출력**: 최종 판정 및 터미널 요약.


In [15]:
canonical_export_ok = (core_fail_count == 0)
reload_ok = (n_fail == 0)
expected_counts_ok = (
    len(registry_src) == 42 and len(pages_df) == 4495 and len(block_df) == 293717
    and len(turn_df_raw) == 65590 and len(segment_df_raw) == 196686
)
hierarchy_ok = all(v == 0 for v in hierarchy_metrics.values())
segment_ok = (segment_fk_failures == 0 and traceability_rate == 1.0 and total_expected == total_observed)

if not (canonical_export_ok and expected_counts_ok and hierarchy_ok and segment_ok and reload_ok):
    final_gate = "FAIL"
elif not _sklearn_ok:
    final_gate = "CANONICAL_ETL_PASS"  # sklearn absence is explicitly INFO-only, not a blocker (req #9)
else:
    final_gate = "CANONICAL_ETL_PASS"

main_blockers = []
if not canonical_export_ok:
    main_blockers.append("quality_audit core_fail_count > 0 -- canonical export was aborted")
if not reload_ok:
    main_blockers.append(f"{n_fail} reload verification checks failed")
if not expected_counts_ok:
    main_blockers.append("row counts do not match the Phase 1-derived expected baseline")

print(f"canonical_export_ok={canonical_export_ok} expected_counts_ok={expected_counts_ok} "
      f"hierarchy_ok={hierarchy_ok} segment_ok={segment_ok} reload_ok={reload_ok}")
print(f"FINAL GATE: {final_gate}")

print()
print("[SECTION 12 RESULT]")
print("status: PASS (gate computed)")
print(f"final_gate: {final_gate}")


canonical_export_ok=True expected_counts_ok=True hierarchy_ok=True segment_ok=True reload_ok=True
FINAL GATE: CANONICAL_ETL_PASS

[SECTION 12 RESULT]
status: PASS (gate computed)
final_gate: CANONICAL_ETL_PASS


## 최종 터미널 요약

In [16]:
print("# Canonical ETL Freeze Result (Phase 2)")
print()
print(f"- project_root: {PROJECT_ROOT}")
print(f"- notebook: {NOTEBOOK_PATH.relative_to(PROJECT_ROOT)}")
print(f"- pipeline_run_id: {PIPELINE_RUN_ID}")
print(f"- parser_version: {PARSER_VERSION}  pipeline_version: {PIPELINE_VERSION}")
print(f"- registry_rows: {len(registry_src)}  pdf_files: {len(actual_pdfs)}")
print(f"- pages: {len(pages_df)}  blocks: {len(block_df)}  turns: {len(turn_df_raw)}  segments: {len(segment_df_raw)}")
print(f"- hierarchy_metrics: {hierarchy_metrics}")
print(f"- non_contribution_check: empty_chars={empty_contributes_chars} page_header_turn={page_header_has_turn} "
      f"page_footer_turn={page_footer_has_turn}")
print(f"- segment_fk_failures: {segment_fk_failures}  traceability_rate: {traceability_rate}")
print(f"- expected_segment_count: {total_expected}  observed_segment_count: {total_observed}")
print(f"- quality_audit core_fail_count: {core_fail_count}")
print(f"- sklearn_available: {_sklearn_ok} (INFO only, not a Phase 2 blocker)")
print(f"- reload_checks_failed: {n_fail} / {len(reload_checks)}")
print(f"- canonical_exported: {canonical_export_ok}")
print(f"- main_blockers: {main_blockers}")
print(f"- FINAL GATE: {final_gate}")
print()
print("## Outputs")
for name, p in CANONICAL_PATHS.items():
    print(f"- {name}: {p.relative_to(PROJECT_ROOT)}")
print(f"- sqlite: {SQLITE_PATH.relative_to(PROJECT_ROOT)}")
print(f"- manifest: {MANIFEST_PATH.relative_to(PROJECT_ROOT)}")
print(f"- ssot_doc: {SSOT_PATH.relative_to(PROJECT_ROOT)}")

print()
print(json.dumps({
    "project": "P3_CULTURE", "stage": "CANONICAL_ETL_FREEZE", "pipeline_run_id": PIPELINE_RUN_ID,
    "parser_version": PARSER_VERSION, "pipeline_version": PIPELINE_VERSION,
    "registry_rows": len(registry_src), "pdf_files": len(actual_pdfs),
    "pages": len(pages_df), "blocks": len(block_df), "turns": len(turn_df_raw), "segments": len(segment_df_raw),
    "hierarchy_metrics": hierarchy_metrics, "segment_fk_failures": segment_fk_failures,
    "traceability_rate": traceability_rate, "expected_segment_count": total_expected,
    "observed_segment_count": total_observed, "core_fail_count": core_fail_count,
    "sklearn_available": _sklearn_ok, "reload_checks_failed": n_fail, "canonical_exported": canonical_export_ok,
    "final_gate": final_gate, "main_blockers": main_blockers,
}, ensure_ascii=False, indent=2, default=str))


# Canonical ETL Freeze Result (Phase 2)

- project_root: /home/sieg/projects-wsl/SBS_dataScience/DSJA/P3_CULTURE
- notebook: 09_audit_minutes_pdf_etl.ipynb
- pipeline_run_id: 1.0.1_20260721T084913Z
- parser_version: 1.0.1  pipeline_version: 1.0.1
- registry_rows: 42  pdf_files: 42
- pages: 4495  blocks: 293717  turns: 65590  segments: 196686
- hierarchy_metrics: {'inheritance_fail': 0, 'containment_fail': 0, 'adjacency_fail': 0, 'atomicity_fail': 0, 'n_blocks_mismatch': 0, 'char_count_mismatch': 0, 'page_range_mismatch': 0, 'multiple_speaker_headers': 0, 'speaker_turn_without_header': 0}
- non_contribution_check: empty_chars=0 page_header_turn=0 page_footer_turn=0
- segment_fk_failures: 0  traceability_rate: 1.0
- expected_segment_count: 196686  observed_segment_count: 196686
- quality_audit core_fail_count: 0
- sklearn_available: False (INFO only, not a Phase 2 blocker)
- reload_checks_failed: 0 / 39
- canonical_exported: True
- main_blockers: []
- FINAL GATE: CANONICAL_ETL_PASS

## O